# 01 · End to end: one request through every layer

Phase 0 proved the stack runs. This notebook shows **where the time goes** when a request travels through it, first for single requests and then for a real multi-step agent task.

All numbers are read from `results/01_end_to_end/`.

## Where each number is measured

```
client ──► gateway ──► NGINX ──► vLLM API server ──► engine: queue → prefill → decode
  │           │           │              │                         │
  │           │           │              │                         └ vLLM /metrics (before/after snapshot)
  │           │           │              └ NGINX upstream time minus vLLM end to end
  │           │           └ NGINX access log (rt, urt)
  │           └ gateway JSON log (ttfb_ms, total_ms)
  └ client clock (TTFT, end to end)
```

vLLM only exposes totals, not per-request numbers. Sending one request at a time and snapshotting `/metrics` before and after gives that single request's exact queue, prefill and decode times (`loadtest/vllm_metrics.py`).

Each layer's share is the difference between nested timers: for example, **gateway** = gateway total − NGINX total. Clock resolution (NGINX logs to the millisecond) means very small layers can come out at ±1 ms.

In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(ROOT / "loadtest"))
from results_logger import load_runs, summarize

runs = load_runs("01_end_to_end")
traces = [r for r in runs if r["name"] == "trace_single_requests"]
agent_runs = [r for r in runs if r["name"] == "agent_run"]
print(f"{len(traces)} trace run(s), {len(agent_runs)} agent run(s)")
if runs:
    env = runs[-1]["environment"]
    gpu = env["gpus"][0] if env["gpus"] else {"name": "no GPU", "memory_mib": 0}
    print(f"hardware: {gpu['name']} ({gpu['memory_mib'] / 1024:.0f} GB) | model: {runs[-1]['config'].get('model')} "
          f"| vLLM: {runs[-1]['config'].get('vllm_image')} | config: {runs[-1]['config'].get('vllm_config')}")

## Single requests, layer by layer

Three prompts from `loadtest/prompts.json`, each sent 5 times with nothing else running:

* **short_qa_05**: short question, short answer
* **summarize_01**: ~550 token document, medium answer (prefill heavy)
* **long_gen_02**: short prompt, long answer (decode heavy)

In [ ]:
LAYERS = [
    ("client_to_gateway_s", "client ↔ gateway"),
    ("gateway_s", "gateway"),
    ("nginx_s", "NGINX"),
    ("vllm_api_server_s", "vLLM API server"),
    ("engine_queue_s", "engine: queue"),
    ("engine_prefill_s", "engine: prefill"),
    ("engine_decode_s", "engine: decode"),
    ("engine_other_s", "engine: other"),
]

if not traces:
    print("No trace runs yet: run scripts/run_phase1.sh on the GPU box, then copy results/ back.")
else:
    trace = traces[-1]
    samples = trace["metrics"]["samples"]
    prompt_ids = list(trace["metrics"]["summary"])

    print(f"{'median per request':<20}" + "".join(f"{p:>16}" for p in prompt_ids))
    for key in ["prompt_tokens", "output_tokens"]:
        print(f"{key.replace('_', ' '):<20}" + "".join(
            f"{summarize([s['client'][key] for s in samples if s['prompt_id'] == p])['p50']:>16}" for p in prompt_ids))
    for key, label in [("ttft_s", "TTFT"), ("e2e_s", "end to end")]:
        print(f"{label:<20}" + "".join(
            f"{trace['metrics']['summary'][p][key]['p50'] * 1000:>13.0f} ms" for p in prompt_ids))
    print()
    for key, label in LAYERS:
        values = [trace["metrics"]["summary"][p]["layers_p50_s"][key] for p in prompt_ids]
        print(f"{label:<20}" + "".join(f"{v * 1000:>13.1f} ms" if v is not None else f"{'n/a':>16}" for v in values))

In [ ]:
if traces:
    fig, ax = plt.subplots(figsize=(11, 3.2))
    colors = ["#c7c7c7", "#9ecae1", "#6baed6", "#4292c6", "#fdd0a2", "#fd8d3c", "#d94801", "#bdbdbd"]
    for row, p in enumerate(prompt_ids):
        left = 0.0
        for (key, label), color in zip(LAYERS, colors):
            value = max(trace["metrics"]["summary"][p]["layers_p50_s"][key] or 0.0, 0.0) * 1000
            ax.barh(row, value, left=left, color=color, label=label if row == 0 else None)
            left += value
    ax.set_yticks(range(len(prompt_ids)), prompt_ids)
    ax.invert_yaxis()
    ax.set_xlabel("milliseconds (median of 5 requests)")
    ax.set_title("Where one request's time goes")
    ax.legend(ncol=4, loc="upper center", bbox_to_anchor=(0.5, -0.3), frameon=False)
    plt.tight_layout()
    plt.show()

### Prefill vs decode speed

Prefill processes every prompt token in parallel; decode generates output tokens one at a time. Dividing tokens by time shows how different the two phases are.

In [ ]:
if traces:
    print(f"{'':<14}{'prefill tokens/s':>18}{'decode tokens/s':>18}{'ms per output token':>22}")
    for p in prompt_ids:
        rows = [s for s in samples if s["prompt_id"] == p]
        prefill = summarize([s["client"]["prompt_tokens"] / s["engine"]["prefill_s"] for s in rows if s["engine"]["prefill_s"] > 0])
        decode = summarize([(s["client"]["output_tokens"] - 1) / s["engine"]["decode_s"] for s in rows if s["engine"]["decode_s"] > 0])
        print(f"{p:<14}{prefill['p50']:>18,.0f}{decode['p50']:>18,.0f}{1000 / decode['p50']:>22.1f}")

## The agent: one task, several LLM calls

The CrewAI crew in `agent/app.py` summarizes an incident report with three agents in sequence (analyst → writer → reviewer). Every LLM call carries the same `x-run-id`, so the gateway and NGINX logs can be grouped by run.

In [ ]:
if not agent_runs:
    print("No agent runs yet.")
else:
    print(f"{'run':<26}{'total':>9}{'LLM calls':>11}{'prompt tok':>12}{'gen tok':>9}{'prefill':>10}{'decode':>10}")
    for r in agent_runs:
        m, e = r["metrics"], r["metrics"]["engine"]
        print(f"{m['run_id']:<26}{m['total_s']:>8.1f}s{m['llm_calls']:>11}{e['prompt_tokens']:>12.0f}"
              f"{e['generation_tokens']:>9.0f}{e['prefill_s']:>9.2f}s{e['decode_s']:>9.2f}s")

In [ ]:
if agent_runs:
    run = agent_runs[-1]["metrics"]
    print(f"Latest run {run['run_id']}: {run['total_s']:.1f} s\n")
    print("Per task (CrewAI's view):")
    for t in run["tasks"]:
        print(f"  {t['agent']:<22}{t['duration_s']:>7.1f} s")
    print("\nPer LLM call (gateway and NGINX logs):")
    print(f"  {'#':<3}{'gateway total':>15}{'NGINX total':>13}{'vLLM (upstream)':>17}")
    for i, c in enumerate(run["calls"], 1):
        fmt = lambda v: f"{v:.2f} s" if v is not None else "n/a"
        print(f"  {i:<3}{fmt(c.get('gateway_total_s')):>15}{fmt(c.get('nginx_total_s')):>13}{fmt(c.get('upstream_total_s')):>17}")

    in_llm = sum(c.get("gateway_total_s") or 0 for c in run["calls"])
    e = run["engine"]
    print(f"\nTime waiting on the LLM: {in_llm:.1f} s of {run['total_s']:.1f} s ({in_llm / run['total_s']:.0%})")
    print(f"Agent framework overhead (everything else): {run['total_s'] - in_llm:.2f} s")
    print(f"Inside vLLM: prefill {e['prefill_s']:.2f} s, decode {e['decode_s']:.2f} s, queue {e['queue_s'] * 1000:.0f} ms")

In [ ]:
if agent_runs:
    print(agent_runs[-1]["metrics"]["final_summary"])

## Grafana

The dashboard (`observability/grafana/dashboards/llm_inference.json`) is provisioned automatically. Screenshot from the Phase 1 session:

*(added after the GPU session)*

## What happened

*(written after the GPU session, from the numbers above)*

## Phase 1 exit criteria

- [x] CrewAI agent making several LLM calls through the gateway, tagged with a run ID
- [x] Request IDs and run IDs logged at the gateway and NGINX
- [x] GPU metrics exporter and a provisioned Grafana dashboard
- [x] Layer-by-layer tracing of single requests, saved to `results/01_end_to_end/`
- [ ] Phase 1 session on the RTX 5090: traces, agent runs and dashboard captured
- [ ] One request fully explained with per-layer timings